Vector Database

In [41]:
import os

In [42]:
from dotenv import load_dotenv,find_dotenv
env_path = find_dotenv(".env", usecwd=True)
print("Found .env:", env_path or "NOT FOUND")
print("Loaded:", load_dotenv(env_path, override=True))
key_name = "XAI_API_KEY"

Found .env: c:\Users\prsha\Desktop\DataAnalyst\Udemy_Python\.env
Loaded: True


In [43]:
from pathlib import Path
from llama_index.readers.file import PDFReader
loader = PDFReader()
documents = loader.load_data(file=Path('transformers.pdf'))
len(documents)


15

Load Data

In [44]:
from pathlib import Path
from llama_index.readers.file import PDFReader

In [45]:
from pathlib import Path
from llama_index.readers.file import PDFReader
loader = PDFReader()
documents = loader.load_data(file=Path('transformers.pdf'))
len(documents)


15

vector store Index

In [46]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)

In [47]:
pwd

'c:\\Users\\prsha\\Desktop\\DataAnalyst\\Udemy_Python\\0-DataIngestParsing\\data'

In [ ]:
Path('index').mkdir(parents=True, exist_ok=True)
index.storage_context.persist(persist_dir="./index")

In [49]:
%whos

Variable               Type                    Data/Info
--------------------------------------------------------
HuggingFaceEmbedding   ModelMetaclass          <class 'llama_index.embed<...>se.HuggingFaceEmbedding'>
PDFReader              ABCMeta                 <class 'llama_index.reade<...>ile.docs.base.PDFReader'>
Path                   type                    <class 'pathlib.Path'>
VectorStoreIndex       ABCMeta                 <class 'llama_index.core.<...>e.base.VectorStoreIndex'>
documents              list                    n=15
embed_model            HuggingFaceEmbedding    model_name='sentence-tran<...>e show_progress_bar=False
env_path               str                     c:\Users\prsha\Desktop\Da<...>Analyst\Udemy_Python\.env
find_dotenv            function                <function find_dotenv at 0x000002DED85299B0>
index                  VectorStoreIndex        <llama_index.core.indices<...>ct at 0x000002DE935AEC10>
key_name               str                     XAI_API

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage

storage_context = StorageContext.from_defaults(persist_dir="./index")
index = load_index_from_storage(storage_context, embed_model=embed_model)

Using ChromeDB

In [52]:

#pip install chromadb

In [53]:
# Install with: %pip install llama-index-vector-stores-chroma

In [ ]:
import chromadb
from llama_index.core import SimpleDirectoryReader, StorageContext, VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore

In [ ]:
documents=SimpleDirectoryReader(input_files=['transformers.pdf']).load_data()
db=chromadb.PersistentClient(path="./chroma_db")
chroma_collection=db.get_or_create_collection("quickstart")

In [ ]:
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore

vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    embed_model=embed_model,
)

Using PineCone Vector DB

In [57]:
# Install with: %pip install llama-index-vector-stores-pinecone

In [1]:
import os
from pinecone import Pinecone

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "").strip()
if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY is missing. Add it to the project .env file and rerun the dotenv cell.")

pc = Pinecone(api_key=PINECONE_API_KEY)

In [4]:
from pinecone import ServerlessSpec
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

index_name = "quickstart"
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
embedding_dimension = len(embed_model.get_text_embedding("dimension check"))
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=embedding_dimension,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
pinecone_index = pc.Index(index_name)

c:\Users\prsha\Desktop\DataAnalyst\Udemy_Python\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
